# UTS Threat Hunting & CTI (TK121) - Demo Analisis Dataset WitFoo Precinct6

**Hipotesis yang diuji**

- **H2** - Ada host di ORG-0004 yang memindai port (T1046) atau mencoba koneksi berulang ke banyak port/host, dan pola ini bisa dibedakan dari deny normal memakai metrik *fan-out* per jendela waktu.
- **H3** - Ada endpoint di ORG-0004 yang mengalami upaya mematikan sensor EDR (T1562.001) / menghapus shadow copy (T1490), dan jejaknya bisa dikorelasikan dengan provenance graph.

**Dataset**: `witfoo/precinct6-cybersecurity` v2.1.0 (parquet, 2.011.674 baris sinyal).

**Alur demo**: unduh data -> verifikasi -> validasi integritas arah flow -> query fan-out (H2) -> query cascade EDR (H3) -> bangun evidence timeline -> chart -> ekspor CSV.
Semua timestamp di notebook ini **UTC**. Output konsol sengaja berbahasa Inggris, penjelasan berbahasa Indonesia.


## 1. Setup environment + unduh data

In [ ]:
!pip -q install duckdb huggingface_hub

In [ ]:
import os, json, warnings
import duckdb, pandas as pd
from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

REPO = "witfoo/precinct6-cybersecurity"
FILES = [
    "signals/signals.parquet",          # live signals, 2.011.674 rows
    "graph/attack_reports.parquet",     # 51.371 incident narratives
    "graph/nodes.parquet",              # provenance graph nodes
    "graph/edges-00000.parquet",        # provenance graph edges
]

paths = {}
for f in FILES:
    paths[f] = hf_hub_download(repo_id=REPO, filename=f, repo_type="dataset")
    print(f"downloaded {f:32} {os.path.getsize(paths[f]) / 1e6:8.1f} MB")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW sig AS SELECT * FROM '{paths['signals/signals.parquet']}'")
con.execute(f"CREATE OR REPLACE VIEW reports AS SELECT * FROM '{paths['graph/attack_reports.parquet']}'")
con.execute(f"CREATE OR REPLACE VIEW nodes AS SELECT * FROM '{paths['graph/nodes.parquet']}'")
con.execute(f"CREATE OR REPLACE VIEW edges AS SELECT * FROM '{paths['graph/edges-00000.parquet']}'")

print("\nrows in signals:", con.sql("SELECT count(*) FROM sig").fetchall()[0][0])

## 2. Verifikasi dataset (angka harus sama dengan model card)

In [ ]:
con.sql("""
SELECT label_binary AS label, count(*) AS rows,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM sig GROUP BY 1 ORDER BY rows DESC
""").df()

In [ ]:
con.sql("""
SELECT stream_name, count(*) AS rows, count(*) FILTER (WHERE label_binary <> 'benign') AS flagged
FROM sig GROUP BY 1 ORDER BY rows DESC LIMIT 10
""").df()

## 3. Validasi integritas: arah flow pada kolom vs payload

Kolom `src_ip`/`dst_ip` di dataset tidak selalu searah dengan isi pesan VPC flow. Query ini mengukur seberapa sering. Kalau tidak divalidasi, agregasi `top src_ip` akan salah untuk sebagian data.

In [ ]:
integrity = con.sql("""
SELECT count(*) AS total_rows,
       count(*) FILTER (WHERE str_split(message_sanitized,' ')[4] = src_ip) AS direction_matches_column,
       count(*) FILTER (WHERE str_split(message_sanitized,' ')[4] = dst_ip) AS direction_inverted,
       round(100.0 * count(*) FILTER (WHERE str_split(message_sanitized,' ')[4] = dst_ip) / count(*), 2) AS pct_inverted
FROM sig WHERE stream_name = 'aws_vpc_flow_log'
""").df()
display(integrity)
print("VPC flow actions:", con.sql("SELECT DISTINCT str_split(message_sanitized,' ')[13] FROM sig WHERE stream_name='aws_vpc_flow_log'").fetchall())
print("=> all parsed src/dst numbers below use the message payload, not the columns")

## 4. H2 - Fan-out port per sumber (5 menit)

Menyentuh >=10 port berbeda dalam 5 menit. Baseline: 94% jendela hanya 1 port (lihat bagian kalibrasi).

In [ ]:
q1 = con.sql("""
WITH p AS (
  SELECT str_split(message_sanitized,' ')[4] AS src,
         str_split(message_sanitized,' ')[5] AS dst,
         str_split(message_sanitized,' ')[6] AS sport,
         CAST(str_split(message_sanitized,' ')[7] AS VARCHAR) AS dport,
         CAST(str_split(message_sanitized,' ')[11] AS BIGINT) AS flow_start
  FROM sig WHERE stream_name = 'aws_vpc_flow_log'
)
SELECT src,
       strftime(timezone('UTC', to_timestamp((flow_start // 300) * 300)), '%Y-%m-%d %H:%M') AS window_utc,
       count(*) AS flows, count(DISTINCT dport) AS dst_ports,
       count(DISTINCT sport) AS src_ports, count(DISTINCT dst) AS dst_hosts
FROM p GROUP BY src, window_utc
HAVING count(DISTINCT dport) >= 10
ORDER BY dst_ports DESC
""").df()
display(q1)
print(f"sources with port fan-out: {len(q1)}")

## 5. H2 - Fan-out sumber: banyak IP ke satu tujuan (10 menit)

Pola terdistribusi: tiap IP hanya 1 port, tapi totalnya puluhan sumber.

In [ ]:
q2 = con.sql("""
WITH p AS (
  SELECT str_split(message_sanitized,' ')[4] AS src,
         str_split(message_sanitized,' ')[5] AS dst,
         CAST(str_split(message_sanitized,' ')[7] AS VARCHAR) AS dport,
         CAST(str_split(message_sanitized,' ')[11] AS BIGINT) AS flow_start
  FROM sig WHERE stream_name = 'aws_vpc_flow_log'
)
SELECT dst,
       strftime(timezone('UTC', to_timestamp((flow_start // 600) * 600)), '%Y-%m-%d %H:%M') AS window_utc,
       count(DISTINCT src) AS sources, count(*) AS flows, count(DISTINCT dport) AS dst_ports
FROM p GROUP BY dst, window_utc
HAVING count(DISTINCT src) >= 20
ORDER BY sources DESC
""").df()
display(q2)

## 6. H2 - Percobaan berulang ke satu port (RPC/lateral movement candidate)

Bukan sweep cepat: percobaan berulang dengan **port sumber acak** dan payload kecil, berlangsung berhari-hari. Ini yang lebih relevan untuk T1021.

In [ ]:
q3 = con.sql("""
WITH p AS (
  SELECT str_split(message_sanitized,' ')[4] AS src,
         str_split(message_sanitized,' ')[5] AS dst,
         str_split(message_sanitized,' ')[6] AS sport,
         CAST(str_split(message_sanitized,' ')[7] AS VARCHAR) AS dport,
         str_split(message_sanitized,' ')[9] AS bytes,
         CAST(str_split(message_sanitized,' ')[11] AS BIGINT) AS flow_start
  FROM sig WHERE stream_name = 'aws_vpc_flow_log'
)
SELECT src, dst, dport, count(*) AS attempts, count(DISTINCT sport) AS src_ports,
       count(DISTINCT bytes) AS distinct_sizes,
       strftime(timezone('UTC', to_timestamp(min(flow_start))), '%Y-%m-%d %H:%M') AS first_seen,
       strftime(timezone('UTC', to_timestamp(max(flow_start))), '%Y-%m-%d %H:%M') AS last_seen
FROM p GROUP BY src, dst, dport
HAVING count(*) >= 50 AND count(DISTINCT sport) >= 10
ORDER BY attempts DESC
""").df()
display(q3)

## 7. Kalibrasi ambang (dasar tuning rule)

Distribusi jumlah port unik per sumber per 5 menit. Ini yang dipakai untuk memilih ambang >=10, bukan angka kira-kira.

In [ ]:
cal = con.sql("""
WITH p AS (
  SELECT str_split(message_sanitized,' ')[4] AS src,
         CAST(str_split(message_sanitized,' ')[7] AS VARCHAR) AS dport,
         CAST(str_split(message_sanitized,' ')[11] AS BIGINT) AS flow_start
  FROM sig WHERE stream_name = 'aws_vpc_flow_log'
), w AS (
  SELECT src, (flow_start // 300) AS win, count(DISTINCT dport) AS ports FROM p GROUP BY src, win
)
SELECT ports, count(*) AS windows, round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM w GROUP BY ports ORDER BY ports
""").df()
display(cal)

## 8. H3 - Cascade tampering EDR (dengan deduplikasi wajib)

58 artifact EDR berisi 76 behavior, tetapi hanya **7 perilaku unik**. Tanpa deduplikasi, temuan dilaporkan 10x lebih besar. Perhatikan juga `device_id`: satu token host Precinct ternyata menaungi dua endpoint.

In [ ]:
q4 = con.sql("""
WITH raw AS (
  SELECT artifact_id, timestamp AS ingest_ts, src_host, username, label_binary,
         CAST(regexp_extract(message_sanitized, 'message=(\\{.*?)\\s*(:::|$)', 1) AS JSON) AS j
  FROM sig WHERE stream_name = 'Crowdstrike Detection'
), b AS (
  SELECT artifact_id, ingest_ts, src_host, username, label_binary,
         unnest(json_extract(j, '$.behaviors[*]')) AS beh
  FROM raw
)
SELECT src_host AS host_token,
       json_extract_string(beh,'$.device_id') AS device_id,
       username AS user,
       json_extract_string(beh,'$.timestamp') AS event_time_utc,
       json_extract_string(beh,'$.technique_id') AS technique,
       json_extract_string(beh,'$.display_name') AS behavior,
       json_extract_string(beh,'$.filename') AS filename,
       json_extract_string(beh,'$.md5') AS md5,
       json_extract_string(beh,'$.pattern_disposition_details.fs_operation_blocked') AS fs_blocked,
       label_binary
FROM b
QUALIFY row_number() OVER (
  PARTITION BY json_extract_string(beh,'$.device_id'), json_extract_string(beh,'$.behavior_id'),
               json_extract_string(beh,'$.timestamp'), json_extract_string(beh,'$.cmdline')
  ORDER BY ingest_ts) = 1
ORDER BY event_time_utc
""").df()
display(q4)
print(f"unique EDR behaviors: {len(q4)} | distinct devices: {q4['device_id'].nunique()} | blocked: {(q4['fs_blocked']=='true').sum()}")

## 9. H3 - Bukti bahwa `device_id` berbeda = mesin berbeda

Token `HOST-0023` di dataset menyatukan dua endpoint. Ini memengaruhi kesimpulan: bukan "1 host, 2 user", tetapi "2 endpoint, 2 user".

In [ ]:
dev = con.sql("""
WITH raw AS (
  SELECT CAST(regexp_extract(message_sanitized, 'message=(\\{.*?)\\s*(:::|$)', 1) AS JSON) AS j
  FROM sig WHERE stream_name = 'Crowdstrike Detection'
)
SELECT json_extract_string(j,'$.device.device_id') AS device_id,
       json_extract_string(j,'$.device.hostname')   AS edr_hostname,
       json_extract_string(j,'$.device.os_version') AS os,
       string_agg(DISTINCT json_extract_string(j,'$.device.local_ip'), ', ')      AS local_ips,
       string_agg(DISTINCT json_extract_string(j,'$.device.external_ip'), ', ')   AS external_ips,
       string_agg(DISTINCT json_extract_string(j,'$.device.mac_address'), ', ')   AS macs,
       count(*) AS artifacts
FROM raw
GROUP BY 1,2,3 ORDER BY artifacts DESC
""").df()
display(dev)

## 10. Evidence timeline gabungan (H2 + H3)

Waktu kejadian diambil dari payload (`behaviors[].timestamp` untuk EDR, `start` untuk VPC flow), **bukan** kolom `signals.timestamp` yang merupakan waktu ingest.

In [ ]:
import datetime as dt

flow = con.sql("""
  SELECT CAST(str_split(message_sanitized,' ')[11] AS BIGINT) AS flow_start,
         str_split(message_sanitized,' ')[4] AS src,
         str_split(message_sanitized,' ')[5] AS dst,
         CAST(str_split(message_sanitized,' ')[7] AS VARCHAR) AS dport
  FROM sig WHERE stream_name = 'aws_vpc_flow_log'
""").df()

rows = []
for _, r in q4.iterrows():
    rows.append({
        "event_time_utc": str(r["event_time_utc"]).replace("Z", "").replace("T", " "),
        "hypothesis": "H3",
        "source": "Crowdstrike Falcon (EDR)",
        "host_user": f"{r['device_id'][:8]} / {r['user']}",
        "activity": f"{r['behavior']} ({r['technique']})",
        "evidence": f"proc={r['filename']} md5={r['md5']} fs_blocked={r['fs_blocked']}",
        "dataset_label": r["label_binary"],
    })

def add_flow(t0, t1, src_names, dst, dport_count, n_flows, note):
    rows.append({
        "event_time_utc": dt.datetime.utcfromtimestamp(int(t0)).strftime("%Y-%m-%d %H:%M:%S"),
        "hypothesis": "H2",
        "source": "AWS VPC Flow Log",
        "host_user": f"{src_names} -> {dst}",
        "activity": f"{note}: {n_flows} REJECT flows, {dport_count} dst ports",
        "evidence": f"flow start={dt.datetime.utcfromtimestamp(int(t0)).strftime('%H:%M:%S')} end={dt.datetime.utcfromtimestamp(int(t1)).strftime('%H:%M:%S')} UTC",
        "dataset_label": "mixed",
    })

def window(a, b, dst, note, srcs_mode, src=None):
    d = flow[(flow.flow_start >= a) & (flow.flow_start < b) & (flow.dst == dst)]
    if src:
        d = d[d.src == src]
    if len(d) == 0:
        return
    srcs = sorted(d.src.unique())
    label = srcs[0] if srcs_mode == "one" else f"{len(srcs)} sources ({srcs[0]}..{srcs[-1]})"
    add_flow(d.flow_start.min(), d.flow_start.max() + 30, label, dst, d.dport.nunique(), len(d), note)

# S2: 2024-07-29 20:35-20:41 UTC - the distributed burst, then the single busiest source
window(1722285357, 1722285960, "10.98.236.105", "distributed scan burst", "many")
window(1722285357, 1722285960, "10.98.236.105", "port fan-out from the busiest source", "one", src="100.64.27.187")
# S1: 2024-07-26 11:19-11:25 UTC
window(1721992754, 1721993111, "10.189.183.176", "scan burst", "many")
# S3: RPC retries
d3 = flow[(flow.src == "10.73.148.8") & (flow.dport == "135")]
if len(d3):
    add_flow(d3.flow_start.min(), d3.flow_start.max(), "10.73.148.8", "10.19.72.193:135", 1, len(d3),
             "repeated RPC attempts (5 days)")

timeline = pd.DataFrame(rows).sort_values("event_time_utc").reset_index(drop=True)
display(timeline)

# merge H3 rows that share the same device/time (dedupe already done in q4)
timeline.to_csv("evidence_timeline.csv", index=False)
print("saved evidence_timeline.csv")

## 11. Chart 1 - volume flow per jam (menemukan burst)

Baseline jam normal 1-30 flow. Dua lonjakan besar = kampanye scan.

In [ ]:
import matplotlib.pyplot as plt

hourly = con.sql("""
SELECT strftime(timezone('UTC', to_timestamp((CAST(str_split(message_sanitized,' ')[11] AS BIGINT) // 3600) * 3600)), '%m-%d %H') AS hour_utc,
       count(*) AS flows, count(DISTINCT str_split(message_sanitized,' ')[4]) AS sources
FROM sig WHERE stream_name = 'aws_vpc_flow_log' GROUP BY 1 ORDER BY 1
""").df()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(hourly["hour_utc"], hourly["flows"], color="#2b6cb0")
ax.set_title("VPC flow REJECT per hour (ORG-0004) - baseline 1-30 vs burst 9.000+")
ax.set_ylabel("flows")
ax.tick_params(axis="x", rotation=90)
for i, v in enumerate(hourly["flows"]):
    if v > 1000:
        ax.annotate(str(v), (i, v), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()
print(hourly.sort_values("flows", ascending=False).head(3).to_string(index=False))

## 12. Chart 2 - port fan-out per sumber (diskriminasi pemindai vs host normal)

197 dari 215 sumber hanya menyentuh 1 port. Pemindai menyentuh puluhan.

In [ ]:
fan = con.sql("""
WITH p AS (
  SELECT str_split(message_sanitized,' ')[4] AS src,
         CAST(str_split(message_sanitized,' ')[7] AS VARCHAR) AS dport
  FROM sig WHERE stream_name = 'aws_vpc_flow_log'
)
SELECT src, count(*) AS flows, count(DISTINCT dport) AS ports
FROM p GROUP BY src HAVING count(DISTINCT dport) > 2 ORDER BY ports DESC
""").df()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(fan["src"], fan["ports"], color="#c05621")
ax.set_title("Sources touching >2 distinct destination ports (VPC flow, ORG-0004)")
ax.set_xlabel("distinct dst ports")
ax.invert_yaxis()
plt.tight_layout(); plt.show()
display(fan)

## 13. Korelasi ke provenance graph (sumber kedua)

Graph menautkan host tujuan ke insiden. Dua metrik berbeda (EVENT vs INCIDENT_LINK) memperkuat bahwa host ini target berulang, bukan korban sekali.

In [ ]:
graph_deg = con.sql("""
SELECT dst AS node,
       count(*) FILTER (WHERE type = 'EVENT') AS event_edges,
       count(*) FILTER (WHERE type = 'INCIDENT_LINK') AS incident_links,
       count(DISTINCT src) AS distinct_sources
FROM edges WHERE dst LIKE 'ORG-0004/%'
GROUP BY dst ORDER BY incident_links DESC LIMIT 8
""").df()
display(graph_deg)

## 14. Ekspor hasil (bisa diunduh dari Colab)

`q1` fan-out per sumber, `q2` fan-out per tujuan, `q3` percobaan berulang, `q4` perilaku EDR unik, `timeline` evidence timeline.

In [ ]:
exports = {"h2_src_fanout.csv": q1, "h2_dst_fanout.csv": q2,
           "h2_repeated_attempts.csv": q3, "h3_edr_behaviors.csv": q4,
           "h3_edr_devices.csv": dev, "evidence_timeline.csv": timeline,
           "threshold_calibration.csv": cal}

for name, df in exports.items():
    df.to_csv(name, index=False)
    print(f"{name:32} {len(df):3} rows")

try:
    from google.colab import files
    for name in ["evidence_timeline.csv", "h3_edr_behaviors.csv", "h2_src_fanout.csv"]:
        files.download(name)
except ImportError:
    print("(not running on Colab - files saved locally in the runtime)")

## 15. Kesimpulan demo (yang diucapkan saat video)

1. **Integritas data dulu**: 17,1% baris VPC flow punya arah kolom terbalik dari payload. Semua analisis memakai payload.
2. **H2 terbukti**: 26 Jul 11:19-11:24 UTC 34 sumber -> 10.189.183.176 (43 port, 2.975 flow); 29 Jul 20:35-20:40 UTC 49 sumber -> 10.98.236.105 (102 port, 9.920 flow). Pola terdistribusi + fan-out satu sumber. Semua REJECT: tidak ada bukti koneksi berhasil.
3. **H3 terbukti (perilaku)**: 5 percobaan tampering sensor EDR pada 2 endpoint (device 3810ea13... dan 6e36ac13...) oleh 2 akun, 19 Jul 2024, semuanya diblokir. 76 behavior mentah = 7 perilaku unik.
4. **Tidak terbukti**: kompromi, persistence, eksfiltrasi, atau identitas aktor. Semua klaim tersebut akan menjadi karangan.
5. **Nilai deteksi**: ambang fan-out dari data (94,4% jendela = 1 port) dan dedupe EDR 15 menit; Precinct sendiri tidak pernah menggabungkan EDR dengan flow (0 insiden lintas sumber).
